# Weather

In [9]:
import torch
from torch.utils.data import Dataset
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from transformers import AutoTokenizer
import ast

    
class WeatherDataset(Dataset):

    def __init__(self, data_dir, tokenizer, max_text_length=2048):
        self.data_dir = Path(data_dir)
        self.tokenizer = tokenizer
        self.max_text_length = max_text_length
        self.samples = []
        self._load_data()

    def _load_data(self):
        df_paths = list(self.data_dir.glob("*.csv"))
        if not df_paths:
            raise ValueError(f"CSV file does not exists under {self.data_dir} directory")
        df_path = df_paths[0]  

        try:
            df = pd.read_csv(df_path)
            df['input_window'] = df['input_window'].apply(self.safe_eval)
            df['output_window'] = df['output_window'].apply(self.safe_eval)
            df['input_timestamps'] = df['input_timestamps'].apply(self.safe_eval)
            df['output_timestamps'] = df['output_timestamps'].apply(self.safe_eval)
            tqdm.write(f"Loaded: {df_path}, {len(df)} original samples in total")
        except Exception as e:
            raise ValueError(f"Load {df_path} fails: {str(e)}")
        
        samples = []
        for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing"):
            try:
                required_fields = ['input_window', 'output_window', 'text', 'input_timestamps', 'output_timestamps', 'past_trend', 'future_trend']
                missing = [f for f in required_fields if f not in df.columns]
                if missing:
                    raise ValueError(f"CSV missing columns: {missing}")
                
                input_window = torch.tensor(row['input_window'], dtype=torch.float32)
                output_window = torch.tensor(row['output_window'], dtype=torch.float32)
                input_timestamps = pd.to_datetime(row['input_timestamps']).astype('int64') / 1e9
                input_timestamps = np.array(input_timestamps, dtype=np.float32)
                output_timestamps = pd.to_datetime(row['output_timestamps']).astype('int64') / 1e9
                output_timestamps = np.array(output_timestamps, dtype=np.float32)
                past_trend = row.get('past_trend', {})
                future_trend = row.get('future_trend', {})

                text_data = row['text']
                text_tokens = self.tokenizer(
                    text_data,
                    max_length=self.max_text_length,
                    padding="max_length",
                    truncation=True,
                    return_tensors="pt"
                )
                input_ids = text_tokens["input_ids"].squeeze(0)
                attention_mask = text_tokens["attention_mask"].squeeze(0)

            except Exception as e:
                tqdm.write(f"Warning: Sample{idx}fails. Error: {str(e)}")
                continue

            samples.append({
                "file_name": row['file_name'],
                "input_timestamps": input_timestamps,
                "output_timestamps": output_timestamps,
                "input_window": input_window,  
                "output_window": output_window,  
                "text_input_ids": input_ids,  
                "text_attention_mask": attention_mask,  
                "input_trend": past_trend,
                "output_trend": future_trend,
            })

        self.samples = samples
        if not self.samples:
            raise ValueError("No valid samples loaded")
        
        input_len = self.samples[0]["input_window"].shape[0]
        output_len = self.samples[0]["output_window"].shape[0]

        tqdm.write(f"\nFinished dataset preparation")
        tqdm.write(f"Number of valid samples: {len(self.samples)}, input_len: {input_len}, output_len: {output_len}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        if idx < 0 or idx >= len(self.samples):
            raise IndexError(f"Index {idx} out of range(total_num: {len(self.samples)})")
        return self.samples[idx]
    
    def safe_eval(self, x):
        if isinstance(x, str):
            return ast.literal_eval(x)
        elif isinstance(x, list):
            return x
        else:
            return ast.literal_eval(str(x))


In [10]:
#data_dir = "aligned_in7days_out1days/weather_short.csv"
data_dir = "./weather/long"
model_path = Path("/Users/zhanglige/Desktop/MoME/llm/Qwen-0.5B")
tokenizer = AutoTokenizer.from_pretrained(model_path)


dataset = WeatherDataset(
    data_dir=data_dir,
    tokenizer=tokenizer,
    max_text_length=1024
)

dataset

Loaded: weather/long/weather_long.csv, 1841 original samples in total


Processing: 100%|██████████| 1841/1841 [00:04<00:00, 382.20it/s]


Finished dataset preparation
Number of valid samples: 1841, input_len: 336, output_len: 72


# Finance

In [23]:
import torch
from torch.utils.data import Dataset
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from transformers import AutoTokenizer
import ast


class FinanceDataset(Dataset):
    def __init__(self, data_dir, tokenizer, max_text_length=2048):  
        self.data_dir = Path(data_dir)
        self.tokenizer = tokenizer
        self.max_text_length = max_text_length
        self.samples = []
        self._load_data()

    def _load_data(self):
        df_paths = list(self.data_dir.glob("*.csv"))
        if not df_paths:
            raise ValueError(f"CSV file does not exists under {self.data_dir} directory")
        df_path = df_paths[0]  

        try:
            df = pd.read_csv(df_path)
            df['input_window'] = df['input_window'].apply(self.safe_eval)
            df['output_window'] = df['output_window'].apply(self.safe_eval)
            df['input_timestamps'] = df['input_timestamps'].apply(self.safe_eval)
            df['output_timestamps'] = df['output_timestamps'].apply(self.safe_eval)
            tqdm.write(f"Loaded: {df_path}, {len(df)} original samples in total")
        except Exception as e:
            raise ValueError(f"Load {df_path} fails: {str(e)}")

        
        samples = []
        for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing"):
            try:
                required_fields = ['input_window', 'output_window', 'text', 'input_timestamps', 
                                   'output_timestamps', 'input_trend', 'output_trend', 'overall_trend']
                missing = [f for f in required_fields if f not in df.columns]
                if missing:
                    raise ValueError(f"CSV missing columns: {missing}")         
                    
                input_window = torch.tensor(row['input_window'], dtype=torch.float32)
                output_window = torch.tensor(row['output_window'], dtype=torch.float32)
                input_timestamps = pd.to_datetime(row['input_timestamps']).astype('int64') / 1e9
                input_timestamps = np.array(input_timestamps, dtype=np.float32)
                output_timestamps = pd.to_datetime(row['output_timestamps']).astype('int64') / 1e9
                output_timestamps = np.array(output_timestamps, dtype=np.float32)
                input_trend = row.get('input_trend', {})
                output_trend = row.get('output_trend', {})
                overall_trend = row.get('overall_trend', {})

                text_data = row['text']
                text_tokens = self.tokenizer(
                    text_data,
                    max_length=self.max_text_length,
                    padding="max_length",
                    truncation=True,
                    return_tensors="pt"
                )
                input_ids = text_tokens["input_ids"].squeeze(0)
                attention_mask = text_tokens["attention_mask"].squeeze(0)

            except Exception as e:
                tqdm.write(f"Warning: Sample{idx}fails. Error: {str(e)}")
                continue

            samples.append({
                "file_name": row['file_name'],
                "input_timestamps": input_timestamps,
                "output_timestamps": output_timestamps,
                "input_window": input_window ,  
                "output_window": output_window,  
                "text_input_ids": input_ids,  
                "text_attention_mask": attention_mask,  
                "input_trend": input_trend,
                "output_trend": output_trend,
                "overall_trend": overall_trend,
            })

        self.samples = samples
        if not self.samples:
            raise ValueError("No valid samples loaded")

        input_len = self.samples[0]["input_window"].shape[0]
        output_len = self.samples[0]["output_window"].shape[0]

        tqdm.write(f"\nFinished dataset preparation")
        tqdm.write(f"Number of valid samples: {len(self.samples)}, input_len: {input_len}, output_len: {output_len}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        if idx < 0 or idx >= len(self.samples):
            raise IndexError(f"Index {idx} out of range(total_num: {len(self.samples)})")
        return self.samples[idx]
    
    def safe_eval(self, x):
        if isinstance(x, str):
            return ast.literal_eval(x)
        elif isinstance(x, list):
            return x
        else:
            return ast.literal_eval(str(x))

In [26]:
#data_dir = "aligned_in7days_out1days/weather_short.csv"
data_dir = "./finance/long"
model_path = Path("/Users/zhanglige/Desktop/MoME/llm/Qwen-0.5B")
tokenizer = AutoTokenizer.from_pretrained(model_path)

dataset = FinanceDataset(
    data_dir=data_dir,
    tokenizer=tokenizer,
    max_text_length=1024
)

dataset

Loaded: finance/long/finance_long.csv, 1458 original samples in total


Processing: 100%|██████████| 1458/1458 [00:04<00:00, 299.32it/s]


Finished dataset preparation
Number of valid samples: 1458, input_len: 134, output_len: 33


# No Longer Used

In [ ]:
import pandas as pd

# past
data = pd.read_parquet("./weather/aligned_in14days_out3days/weather_long.parquet")
data = data[['file_name','input_window', 'output_window', 'text', 'input_timestamps', 'output_timestamps', 'past_trend', 'future_trend']]
df = data
df.to_csv("weather_long.csv", index=False)


test = pd.read_csv("weather_long.csv")
test

,file_name,input_window,output_window,text,input_timestamps,output_timestamps,past_trend,future_trend
0,USW00013874_34.json,"[18.9, 17.966666666666665, 17.03333333333333, ...","[13.3, 13.200000000000001, 13.1, 13.0, 13.0, 1...","The following events were reported: Hail, Ligh...","['2011-03-13 00:00:00', '2011-03-13 01:00:00',...","['2011-03-27 00:00:00', '2011-03-27 01:00:00',...",stable,decreasing
1,USW00014739_10.json,"[21.7, 20.933333333333334, 20.166666666666664,...","[12.8, 13.166666666666668, 13.533333333333333,...",The following events were reported: Thundersto...,"['2020-09-24 00:00:00', '2020-09-24 01:00:00',...","['2020-10-08 00:00:00', '2020-10-08 01:00:00',...",decreasing,increasing
2,USW00014820_27.json,"[28.9, 27.03333333333333, 25.166666666666668, ...","[28.3, 27.2, 26.1, 25.0, 23.9, 22.8, 21.7, 21....",The following events were reported: Flash Floo...,"['2011-07-06 00:00:00', '2011-07-06 01:00:00',...","['2011-07-20 00:00:00', '2011-07-20 01:00:00',...",stable,increasing
3,USW00094846_24.json,"[10.0, 9.9, 9.8, 9.7, 9.6, 9.5, 9.4, 8.9500000...","[24.4, 24.4, 24.4, 24.4, 24.4, 24.4, 24.4, 23....",The following events were reported: Flash Floo...,"['2006-09-20 00:00:00', '2006-09-20 01:00:00',...","['2006-10-04 00:00:00', '2006-10-04 01:00:00',...",increasing,decreasing
4,USW00094789_20.json,"[2.8, 4.266666666666667, 5.733333333333333, 7....","[4.4, 4.033333333333333, 3.6666666666666665, 3...",The following events were reported: Heavy Rain...,"['2017-01-11 00:00:00', '2017-01-11 01:00:00',...","['2017-01-25 00:00:00', '2017-01-25 01:00:00',...",stable,stable
...,...,...,...,...,...,...,...,...
1836,USW00013904_4.json,"[33.3, 31.1, 29.4, 26.1, 24.4, 26.1, 22.8, 22....","[35.0, 32.8, 31.1, 26.0, 27.2, 26.7, 27.8, 27....","The following events were reported: Hail, Thun...","['2023-09-11 00:53:00', '2023-09-11 01:53:00',...","['2023-09-25 00:53:00', '2023-09-25 01:53:00',...",increasing,decreasing
1837,USW00094847_36.json,"[20.6, 20.033333333333335, 19.466666666666665,...","[21.1, 20.933333333333334, 20.76666666666667, ...","The following events were reported: Hail, Thun...","['2006-06-08 00:00:00', '2006-06-08 01:00:00',...","['2006-06-22 00:00:00', '2006-06-22 01:00:00',...",increasing,decreasing
1838,USW00094789_7.json,"[23.9, 23.333333333333332, 22.766666666666666,...","[22.8, 21.5, 20.2, 18.9, 18.9, 18.9, 18.9, 18....",The following events were reported: Thundersto...,"['2012-08-26 00:00:00', '2012-08-26 01:00:00',...","['2012-09-09 00:00:00', '2012-09-09 01:00:00',...",stable,decreasing
1839,USW00023293_38.json,"[13.3, 12.8, 12.8, 12.2, 12.2, 11.1, 11.1, 11....","[13.3, 12.8, 11.7, 10.6, 10.0, 10.0, 10.0, 10....",The following events were reported: Flood. The...,"['2005-12-18 00:53:00', '2005-12-18 01:53:00',...","['2006-01-01 00:53:00', '2006-01-01 01:53:00',...",stable,decreasing


In [25]:
import pandas as pd

# past
data = pd.read_parquet("./finance/long/finance_long.parquet")
df = data.rename(columns={'text.content': 'text', 'trend.input_bin_label': 'input_trend', 'trend.output_bin_label': 'output_trend', 'trend.overall_bin_label': 'overall_trend'})

df = df[['file_name', 'input_window', 'output_window', 'text', 'input_timestamps', 'output_timestamps', 'input_trend', 'output_trend', 'overall_trend']]
df.to_csv("./finance/long/finance_long.csv", index=False)

test = pd.read_csv("./finance/long/finance_long.csv")
test

,file_name,input_window,output_window,text,input_timestamps,output_timestamps,input_trend,output_trend,overall_trend
0,20_FANG.json,"[143.5, 143.1, 143.76, 142.0, 142.9, 144.3, 14...","[134.03, 134.775, 135.08, 134.91, 134.6431, 13...",Can Value Investors Select Diamondback Energy ...,"[1677798000.0, 1677801600.0, 1677805200.0, 167...","[1680284100.0, 1680287700.0, 1680291300.0, 168...",<-4%,>+4%,-2% ~ +2%
1,6347_INTC.json,"[25.33, 25.395, 25.77, 26.06, 26.2, 26.24, 26....","[31.8, 32.41, 32.655, 32.765, 32.795, 32.74, 3...",Intel Corporation (INTC) is Attracting Investo...,"[1677794400.0, 1677798000.0, 1677801600.0, 167...","[1680283800.0, 1680287400.0, 1680291000.0, 168...",>+4%,+2% ~ +4%,>+4%
2,1025_TGT.json,"[130.8, 130.76, 130.9, 131.7975, 132.06, 131.9...","[128.52, 129.445, 129.5, 129.4, 128.25, 132.31...","Here's why Target removed Pride-related items,...","[1689718500.0, 1689722100.0, 1689788700.0, 168...","[1692215100.0, 1692218700.0, 1692222300.0, 169...",-2% ~ +2%,-2% ~ +2%,<-4%
3,3550_ROST.json,"[99.17, 100.28, 99.29, 99.36, 96.48, 96.32, 95...","[82.92, 82.56, 81.93, 81.59, 81.61, 81.25, 81....",Why Ross Stores Lost 15% in May\nBy newsfeedba...,"[1651783500.0, 1651787100.0, 1651790700.0, 165...","[1654536600.0, 1654540200.0, 1654543800.0, 165...",<-4%,-2% ~ -4%,<-4%
4,7951_PEP.json,"[170.27, 169.19, 168.05, 168.44, 168.58, 168.4...","[170.175, 169.76, 169.34, 168.86, 169.21, 169....",September PPI Data Higher-Than-Expected\nBy Za...,"[1663104300.0, 1663107900.0, 1663111500.0, 166...","[1665600900.0, 1665604500.0, 1665608100.0, 166...",<-4%,-2% ~ +2%,-2% ~ +2%
...,...,...,...,...,...,...,...,...,...
1453,2175_IVR.json,"[30.25, 29.962, 29.8, 29.8, 29.75, 29.95, 30.0...","[28.5, 28.6, 28.7, 28.95, 28.95, 28.95, 28.85,...",Is the Options Market Predicting a Spike in In...,"[1638304200.0, 1638307800.0, 1638311400.0, 163...","[1640822400.0, 1640826000.0, 1640892600.0, 164...",-2% ~ -4%,-2% ~ +2%,-2% ~ -4%
1454,7056_JPM.json,"[157.56, 157.2, 157.07, 156.63, 156.72, 156.84...","[164.07, 161.23, 161.42, 161.31, 161.07, 160.9...",JPMorgan Chase & Co. (JPM) Surpasses Q3 Earnin...,"[1631649600.0, 1631653200.0, 1631656800.0, 163...","[1634146200.0, 1634149800.0, 1634153400.0, 163...",+2% ~ +4%,-2% ~ +2%,>+4%
1455,609_SLG.json,"[77.22, 76.6941, 76.8745, 76.725, 77.9831, 75....","[71.9503, 71.3006, 71.2594, 71.6822, 72.1875, ...",SL Green Realty Corp. Announces Common Stock D...,"[1626984000.0, 1626987600.0, 1626991200.0, 162...","[1629480600.0, 1629484200.0, 1629487800.0, 162...",<-4%,-2% ~ +2%,<-4%
1456,5409_ABBV.json,"[140.95, 140.8, 140.28, 140.251, 138.625, 138....","[142.15, 143.21, 142.49, 142.9, 142.95, 141.84...",Is Trending Stock AbbVie Inc. (ABBV) a Buy Now...,"[1661200200.0, 1661203800.0, 1661207400.0, 166...","[1663781400.0, 1663785000.0, 1663788600.0, 166...",-2% ~ +2%,-2% ~ +2%,-2% ~ +2%
